# RPS PreTrained YOLO Fine-tuning

## g-drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## [단계 1: 라이브러리 설치 및 Roboflow 데이터셋 다운로드]

In [ ]:
# 1. 필요한 라이브러리 설치 (YOLO 및 Roboflow)
!pip -q install -U ultralytics
!pip -q install roboflow

from roboflow import Roboflow

# 2. Roboflow 데이터셋 다운로드
rf = Roboflow(api_key="8w58ppDvG94zf2wwLw8o")
project = rf.workspace("-gsrkl").project("navigation-wadza")
version = project.version(1)
dataset = version.download("yolov11")

# 3. 다운로드된 데이터셋의 최상위 경로 확인
print("데이터셋 위치:", dataset.location)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 32.8 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to navigation-1 in yolov11:: 100%|██████████| 8933/8933 [00:01<00:00, 6256.35it/s]


데이터셋 위치: /content/navigation-1


## [단계 2: (선택) 클래스 정보 및 yaml 파일 확인]

In [ ]:
import yaml

# Roboflow가 자동 생성한 data.yaml 내용 확인
yaml_path = f"{dataset.location}/data.yaml"

with open(yaml_path, 'r') as f:
    data_yaml = yaml.safe_load(f)

print(data_yaml)

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 7, 'names': ['Bollard', 'bin', 'crosswalk', 'downstairs', 'greenlight', 'redlight', 'upstairs'], 'roboflow': {'workspace': '-gsrkl', 'project': 'navigation-wadza', 'version': 1, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/-gsrkl/navigation-wadza/dataset/1'}}


## [단계 3: YOLO 모델 학습 진행]

In [ ]:
from ultralytics import YOLO

# 1. 구글 드라이브 마운트 (학습 결과 저장을 위해 필요)
from google.colab import drive
drive.mount('/content/drive')

# 2. 모델 인스턴스 생성
yolo = YOLO('yolo11n.pt')

# 3. 모델 학습 시작
results = yolo.train(
    data=f"{dataset.location}/data.yaml", # 기존 DATA 변수 대신 Roboflow 경로 사용
    epochs=80,
    batch=16,
    imgsz=640,
    device=0,
    workers=4,
    project='/content/drive/MyDrive/AI_COLAB_examples/runs', # 구글 드라이브 내 학습 결과 저장 폴더
    name='obstacle_roboflow_v1',
    degrees=15.0,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.4.78 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/navigation-1/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, e

## ONNX 모델로 변환

In [ ]:
import os
from google.colab import files

# 1. 학습 완료 후 저장된 best.pt 경로 지정
project_root = '/content/drive/MyDrive/AI_COLAB_examples/runs'
run_name = 'obstacle_roboflow_v1'
best_model_path = f"{project_root}/{run_name}/weights/best.pt"

# 2. 가중치 불러오기 및 ONNX 변환
best_model = YOLO(best_model_path)
best_model.export(
    format='onnx',
    imgsz=640,
    opset=18,
    simplify=True
)

# 3. 변환된 ONNX 파일 경로
onnx_file_path = f"{project_root}/{run_name}/weights/best.onnx"

# 4. 로컬 PC로 다운로드
if os.path.exists(onnx_file_path):
    print("ONNX 변환 완료. 다운로드를 시작합니다.")
    files.download(onnx_file_path)
else:
    print("경로에 ONNX 파일이 존재하지 않습니다.")